
# CycleGAN – Workshop Notebook (Educational / Toy Example)

This notebook is designed for **teaching CycleGAN concepts clearly**, not for production training.

Target audience:
- Undergraduate / Postgraduate Deep Learning students
- Learners who already understand GAN and DCGAN

Key learning objectives:
- Understand unpaired image-to-image translation
- Learn why CycleGAN needs *two generators and two discriminators*
- Understand **cycle consistency loss**
- See how multi-loss GAN training works



## 1. What Problem Does CycleGAN Solve?

CycleGAN performs **image-to-image translation without paired data**.

Example:
- Domain A: handwritten digits
- Domain B: inverted handwritten digits

We do **not** need matching pairs (A, B).



## 2. High-Level Architecture

CycleGAN contains:
- Generator G_AB : A → B
- Generator G_BA : B → A
- Discriminator D_A
- Discriminator D_B

Key constraint:
If we map A → B → A, we should get back the original image.
This is called **cycle consistency**.



## 3. Imports and Setup


In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt



## 4. Toy Dataset Definition

We simulate two domains using MNIST:
- Domain A: normal MNIST images
- Domain B: inverted MNIST images

This is ideal for learning CycleGAN mechanics.


In [ ]:

transform_A = transforms.Compose([
    transforms.ToTensor()
])

transform_B = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: 1 - x)
])

dataset_A = datasets.MNIST(
    root="./data",
    train=True,
    transform=transform_A,
    download=True
)

dataset_B = datasets.MNIST(
    root="./data",
    train=True,
    transform=transform_B,
    download=True
)

loader_A = DataLoader(dataset_A, batch_size=64, shuffle=True)
loader_B = DataLoader(dataset_B, batch_size=64, shuffle=True)



## 5. Generator Network (Very Simple)

For educational purposes, we use a **shallow CNN**.


In [ ]:

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 1, 3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)



## 6. Discriminator Network

Binary classifier: real vs fake.


In [ ]:

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Flatten(),
            nn.Linear(32 * 14 * 14, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)



## 7. Model Initialization


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

G_AB = Generator().to(device)
G_BA = Generator().to(device)

D_A = Discriminator().to(device)
D_B = Discriminator().to(device)



## 8. Loss Functions

CycleGAN uses **three losses**:
1. Adversarial loss
2. Cycle consistency loss
3. Identity loss (optional – skipped here for simplicity)


In [ ]:

criterion_gan = nn.BCELoss()
criterion_cycle = nn.L1Loss()



## 9. Optimizers


In [ ]:

optimizer_G = optim.Adam(
    list(G_AB.parameters()) + list(G_BA.parameters()), lr=0.0002
)

optimizer_D_A = optim.Adam(D_A.parameters(), lr=0.0002)
optimizer_D_B = optim.Adam(D_B.parameters(), lr=0.0002)



## 10. Training Loop (One Iteration Explained)

This loop shows **how losses interact**.
We train only a few epochs for demonstration.


In [ ]:

epochs = 5

for epoch in range(epochs):
    for (real_A, _), (real_B, _) in zip(loader_A, loader_B):

        real_A = real_A.to(device)
        real_B = real_B.to(device)

        batch = real_A.size(0)
        real_labels = torch.ones(batch, 1).to(device)
        fake_labels = torch.zeros(batch, 1).to(device)

        # -------- Train Generators --------
        fake_B = G_AB(real_A)
        fake_A = G_BA(real_B)

        loss_GAN_AB = criterion_gan(D_B(fake_B), real_labels)
        loss_GAN_BA = criterion_gan(D_A(fake_A), real_labels)

        rec_A = G_BA(fake_B)
        rec_B = G_AB(fake_A)

        loss_cycle = criterion_cycle(rec_A, real_A) + criterion_cycle(rec_B, real_B)

        loss_G = loss_GAN_AB + loss_GAN_BA + 10 * loss_cycle

        optimizer_G.zero_grad()
        loss_G.backward()
        optimizer_G.step()

        # -------- Train Discriminator A --------
        loss_D_A = (
            criterion_gan(D_A(real_A), real_labels) +
            criterion_gan(D_A(fake_A.detach()), fake_labels)
        )

        optimizer_D_A.zero_grad()
        loss_D_A.backward()
        optimizer_D_A.step()

        # -------- Train Discriminator B --------
        loss_D_B = (
            criterion_gan(D_B(real_B), real_labels) +
            criterion_gan(D_B(fake_B.detach()), fake_labels)
        )

        optimizer_D_B.zero_grad()
        loss_D_B.backward()
        optimizer_D_B.step()

    print(f"Epoch {epoch+1} | G: {loss_G.item():.4f}")



## 11. Visualizing Translation

This helps students *see* cycle consistency.


In [ ]:

G_AB.eval()
G_BA.eval()

with torch.no_grad():
    sample_A, _ = next(iter(loader_A))
    sample_A = sample_A[:5].to(device)
    translated_B = G_AB(sample_A)
    reconstructed_A = G_BA(translated_B)

plt.figure(figsize=(9,3))
for i in range(5):
    plt.subplot(3,5,i+1)
    plt.imshow(sample_A[i][0], cmap="gray")
    plt.axis("off")

    plt.subplot(3,5,i+6)
    plt.imshow(translated_B[i][0], cmap="gray")
    plt.axis("off")

    plt.subplot(3,5,i+11)
    plt.imshow(reconstructed_A[i][0], cmap="gray")
    plt.axis("off")

plt.suptitle("Top: A | Middle: A→B | Bottom: A→B→A")
plt.show()



## 12. Key Learning Outcomes

- CycleGAN learns without paired data
- Two generators enforce invertibility
- Cycle loss stabilizes training
- GAN loss alone is insufficient



## 13. Limitations of This Toy Example

- Not PatchGAN
- No identity loss
- No residual blocks
- Small dataset

But **perfect for understanding**.
